## 13-feature-era.ipynb

Builds the `era_bin` feature and sparse era matrix for every album in the `valid_albums` scope
using an Option D fallback chain across three year sources:

1. `release_group_meta_year` — canonical MusicBrainz first-release year (`mb_release_year.parquet`)
2. `album_country_year` — earliest country-specific release date (`mb_album_country.parquet`)
3. `artist_begin_year` — band formation / artist birth year (`mb_artist.parquet`)

Five known data-entry errors (verified via web search) are corrected before binning.
Years > 2026 are hard-capped to `Unknown`.

**Era bins:** `Pre-1900`, `1900–1949` (50-year buckets for sparse pre-modern data),
then `1950s` → `2020s` in standard decade bins. `Unknown` albums get an all-zero row in
the matrix — no signal, no column.

**Data review:** see `1-EDA/07-EDA-year.ipynb` for coverage analysis, source agreement,
outlier review, and sanity checks.

**Inputs:** `data/mb_release_year.parquet`, `data/mb_album_country.parquet`,
`data/mb_album_artists.parquet`, `data/mb_artist.parquet`, `data/mb_album.parquet`

**Outputs:**
- `data/features/album_era.parquet` — one row per album: `album_id`, `best_year`, `best_year_source`, `era_bin`
- `data/features/album_era_matrix.npz` — sparse one-hot matrix `(n_albums × 12)` aligned to `album_ids.pkl`

In [ ]:
import pandas as pd
import numpy as np

DATA_DIR     = '../data'
FEATURES_DIR = f'{DATA_DIR}/features'

## Load sources

In [ ]:
albums = pd.read_parquet(f'{DATA_DIR}/mb_album.parquet').rename(columns={'id': 'album_id'})

rg_year = pd.read_parquet(f'{DATA_DIR}/mb_release_year.parquet')

country_year = (
    pd.read_parquet(f'{DATA_DIR}/mb_album_country.parquet', columns=['album_id', 'album_year'])
    .rename(columns={'album_year': 'album_country_year'})
)

album_artists = pd.read_parquet(f'{DATA_DIR}/mb_album_artists.parquet', columns=['album_id', 'artist_id', 'artist_name'])
artists = pd.read_parquet(f'{DATA_DIR}/mb_artist.parquet', columns=['id', 'artist_year']).rename(columns={'id': 'artist_id', 'artist_year': 'artist_begin_year'})
artist_year = album_artists.merge(artists, on='artist_id', how='left')[['album_id', 'artist_name', 'artist_begin_year']]

print(f'Albums in scope : {len(albums):,}')
print(f'rg_meta rows    : {len(rg_year):,}')
print(f'country rows    : {len(country_year):,}')
print(f'artist rows     : {len(artist_year):,}')

## Assemble: join all three sources

In [ ]:
df = (
    albums[['album_id', 'name']]
    .merge(rg_year, on='album_id', how='left')
    .merge(country_year, on='album_id', how='left')
    .merge(artist_year, on='album_id', how='left')
)
print(f'Assembled: {len(df):,} rows')

## Known year corrections

Five albums with confirmed bad years (verified via web search). Applied to `best_year`
after the fallback chain so source columns remain unchanged for audit purposes.

| album_id | Album | Artist | Raw year | Corrected | Source |
|---|---|---|---|---|---|
| 4576816 | Signes & Racines | Adag'nan | 2027 | 2021 | Bandcamp / Amazon |
| 4598782 | Original Music Box Melodies Of Christmas | Christmas Music Box | 2069 | 1969 | Discogs (Pickwick vinyl) |
| 4615228 | Meditations, Vol. 1 | Helisir | 2205 | 2025 | Bandcamp |
| 4643359 | diSTILLed | Russ Still | 2202 | 2022 | Artist confirmed real; transposition |
| 4706113 | The Entertainment | The Clockworks | 2027 | 2026 | V2 Records (March 2026) |

In [ ]:
YEAR_CORRECTIONS = {
    4576816: 2021,
    4598782: 1969,
    4615228: 2025,
    4643359: 2022,
    4706113: 2026,
}

## Option D fallback chain + era binning

Priority: `release_group_meta_year` → `album_country_year` → `artist_begin_year`.
Known corrections applied after derivation. Years > 2026 hard-capped to `Unknown`.

In [ ]:
df['best_year'] = (
    df['release_group_meta_year']
    .combine_first(df['album_country_year'])
    .combine_first(df['artist_begin_year'])
)
df['best_year_source'] = np.select(
    [
        df['release_group_meta_year'].notna(),
        df['album_country_year'].notna(),
        df['artist_begin_year'].notna(),
    ],
    ['release_group_meta', 'album_country', 'artist_begin'],
    default='unknown'
)

for album_id, corrected_year in YEAR_CORRECTIONS.items():
    mask = df['album_id'] == album_id
    old  = df.loc[mask, 'best_year'].values
    df.loc[mask, 'best_year']        = float(corrected_year)
    df.loc[mask, 'best_year_source'] = 'manual_correction'
    print(f'  album_id={album_id}  {old[0]:.0f} → {corrected_year}')

def assign_era(year):
    if pd.isna(year): return 'Unknown'
    y = int(year)
    if y > 2026: return 'Unknown'
    if y < 1900: return 'Pre-1900'
    if y < 1950: return '1900–1949'
    return f'{(y // 10) * 10}s'

df['era_bin'] = df['best_year'].apply(assign_era)

print(f'\nSource breakdown:')
print(df['best_year_source'].value_counts().to_string())
print(f'\nEra bin counts:')
era_order = ['Pre-1900', '1900–1949'] + [f'{d}s' for d in range(1950, 2030, 10)] + ['Unknown']
print(df['era_bin'].value_counts().reindex(era_order, fill_value=0).to_string())

## Save feature parquet

In [ ]:
out = df[['album_id', 'best_year', 'best_year_source', 'era_bin']]
out.to_parquet(f'{FEATURES_DIR}/album_era.parquet', index=False, compression='zstd')

print(f'Saved: {FEATURES_DIR}/album_era.parquet')
print(f'Shape : {out.shape}')
print(f'Unknown era: {(out["era_bin"] == "Unknown").sum():,}  ({(out["era_bin"] == "Unknown").mean()*100:.1f}%)')

## Build era matrix

One-hot sparse matrix aligned to the master `album_ids.pkl` index, following the same
contract as all other feature matrices. Each column is one era bin; each row is one album.
`Unknown` era albums get an all-zero row (no signal) rather than their own column, so they
don't pollute similarity scores.

Era bins are ordered chronologically so column indices are human-readable:
`Pre-1900`, `1900–1949`, `1950s`, `1960s`, … `2020s`.

In [ ]:
import pickle
from scipy.sparse import csr_matrix, save_npz

# Load master album index — all matrices must align to this
with open(f'{FEATURES_DIR}/album_ids.pkl', 'rb') as f:
    album_ids = pickle.load(f)

album_index = pd.Index(album_ids)
n_albums = len(album_index)
print(f'Master album universe: {n_albums:,}')

In [ ]:
era_parquet = pd.read_parquet(f'{FEATURES_DIR}/album_era.parquet')

# Ordered era vocabulary — Unknown excluded (zero row, not a column)
ERA_ORDER = ['Pre-1900', '1900–1949'] + [f'{d}s' for d in range(1950, 2030, 10)]
era_index = pd.Index(ERA_ORDER)
n_eras = len(era_index)

# Only keep albums with a known era and present in the master index
era_known = era_parquet[era_parquet['era_bin'] != 'Unknown'].copy()

row_idx = album_index.get_indexer(era_known['album_id'].values)
col_idx = era_index.get_indexer(era_known['era_bin'].values)

valid = (row_idx >= 0) & (col_idx >= 0)

X_era = csr_matrix(
    (np.ones(valid.sum(), dtype=np.float32),
     (row_idx[valid], col_idx[valid])),
    shape=(n_albums, n_eras)
)

print(f'Era columns    : {ERA_ORDER}')
print(f'X_era shape    : {X_era.shape}')
print(f'Non-zero rows  : {(X_era.sum(axis=1) > 0).sum():,}  ({(X_era.sum(axis=1) > 0).mean()*100:.1f}%)')
print(f'Zero rows (Unknown era): {(X_era.sum(axis=1) == 0).sum():,}')

## Spot-check: column sums

Confirm the per-era album counts in the matrix match the era distribution from the feature build.

In [ ]:
col_sums = np.asarray(X_era.sum(axis=0)).flatten()
print('Albums per era column:')
for era, count in zip(ERA_ORDER, col_sums):
    print(f'  {era:<12} {int(count):>8,}')

## Save matrix

In [ ]:
save_npz(f'{FEATURES_DIR}/album_era_matrix.npz', X_era)
print(f'Saved : {FEATURES_DIR}/album_era_matrix.npz')
print(f'Shape : {X_era.shape}')
print(f'nnz   : {X_era.nnz:,}')